# 3. StructuredOutputParser (+ ResponseSchema)

The **no-Pydantic, minimal-boilerplate** way to get a dict of named fields out of the model. You list
the fields you want as `ResponseSchema` objects; the parser handles instructions + parsing.

---

## 1. Simple Definition

> **Kid version:** Instead of building a fancy form class, you jot a quick **checklist** of the boxes
> you want — "I need an `answer` and a `source`." This parser turns that checklist into instructions
> for the AI and reads the reply back into a dictionary.

**Professional definition:** `StructuredOutputParser` builds format instructions and parses model
output into a `dict`, using a list of **`ResponseSchema`** items (each = one field with a `name`,
`description`, and optional `type`). No Pydantic model needed.

```python
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name="answer", description="the answer to the question"),
    ResponseSchema(name="source", description="the source used, if any"),
]
parser = StructuredOutputParser.from_response_schemas(schemas)
```

---

## 2. Why Does It Exist?

**The problem:** You want a few named fields (not just a string or list), but pulling in Pydantic feels
heavy for a quick task, and beginners may not know Pydantic yet.

### Before (Pydantic for 2 fields feels like overkill)

```python
class QA(BaseModel):
    answer: str = Field(description="the answer")
    source: str = Field(description="the source")
parser = PydanticOutputParser(pydantic_object=QA)
```

### After (just list the fields)

```python
parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer"),
    ResponseSchema(name="source", description="the source"),
])
```

Less ceremony for simple, flat outputs. Trade-off: **no type validation** (values come back as
strings) and **no nesting** — for those, use `PydanticOutputParser`.

---

## 3. Real-Life Analogy

A **sticky-note checklist** 📝 instead of an official printed form. For "grab the *answer* and the
*source*", a quick two-item note is faster than designing a formal document. The notes are your
`ResponseSchema`s.

---

## 4. Where It Fits in LangChain Architecture

```
BaseOutputParser
    │
    ▼
StructuredOutputParser        ← built from a list of ResponseSchema → dict
        ▲
        │ made of
   ResponseSchema(name, description, type)   ← one per field
```

- `ResponseSchema` = description of **one field**.
- `StructuredOutputParser` = assembles fields into instructions + a dict parser.
- Simpler cousin of `PydanticOutputParser` (dict, no validation, no nesting).

---

## 5. Internal Working

```
  ResponseSchemas: [answer, source]
        │
        ▼
  get_format_instructions()  → a fenced JSON template:
        ```json
        {
          "answer": string  // the answer to the question
          "source": string  // the source used, if any
        }
        ```
        │  (injected into the prompt)
        ▼
  model replies with that JSON (inside ```json fences)
        │
        ▼
  parse(text)  → strip fences, json.loads → dict
        │
        ▼
  {"answer": "...", "source": "..."}
```

---

## 6. Attributes / Methods

### `ResponseSchema(name, description, type)`

**Definition:** Describes one output field: `name` (dict key), `description` (instruction to the
model), optional `type` (e.g. `"string"`, `"list"`; default `"string"`).

**Why it exists:** The minimal unit of "a field I want back."

**When developers use it:** One per desired field.

**Real-life use case:** One line on your sticky-note checklist.

```python
ResponseSchema(name="tags", description="topic tags", type="list")
```

---

### `StructuredOutputParser.from_response_schemas()`

**Definition:** Classmethod that builds the parser from a list of `ResponseSchema`s.

```python
parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer"),
    ResponseSchema(name="confidence", description="0-1 confidence"),
])
```

---

### `get_format_instructions()`

**Definition:** Returns the fenced-JSON template telling the model which keys to produce.

```python
print(parser.get_format_instructions())
```

---

### `parse()`

**Definition:** Converts the model's JSON text into a `dict` (values are strings — no type coercion).

```python
parser.parse('```json\n{"answer":"Paris","confidence":"0.9"}\n```')
# {'answer': 'Paris', 'confidence': '0.9'}   ← note: "0.9" stays a string
```

---

## Putting it together

```python
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer to the user's question"),
    ResponseSchema(name="source", description="the source used, if any"),
])

prompt = PromptTemplate(
    template="Answer the question.\n{format_instructions}\nQuestion: {question}\n",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | ChatOpenAI(model="gpt-4o-mini") | parser
print(chain.invoke({"question": "Capital of France?"}))
# {'answer': 'Paris', 'source': '...'}
```

---

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

schema = [
    ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
    ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
    ResponseSchema(name='fact_3', description='Fact 3 about the topic'),
]

parser = StructuredOutputParser.from_response_schemas(schema)

topic_template = PromptTemplate(
                                template='Give 3 fact about {topic} \n {format_instruction}',
                                input_variables=['topic'],
                                partial_variables={'format_instruction':parser.get_format_instructions()}
                               )

print(topic_template.format(topic='black hole'))

chain = topic_template | llm | parser

llm_result = chain.invoke({'topic':'black hole'})

llm_result

Give 3 fact about black hole 
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"fact_1": string  // Fact 1 about the topic
	"fact_2": string  // Fact 2 about the topic
	"fact_3": string  // Fact 3 about the topic
}
```


{'fact_1': 'Black holes have an event horizon, a boundary beyond which nothing, not even light, can escape their gravitational pull.',
 'fact_2': 'They come in different sizes, including stellar-mass black holes (formed by collapsing stars) and supermassive black holes (found at the centers of galaxies).',
 'fact_3': "Time dilation near a black hole causes time to slow down for an observer close to it compared to someone farther away, as predicted by Einstein's theory of relativity."}